In [1]:
from pathlib import Path
from PIL import Image

# ================= PROJECT ROOT =================
CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD
while PROJECT_ROOT.name != "Vehicle_Damage_Detection":
    PROJECT_ROOT = PROJECT_ROOT.parent

# ================= PATHS =================
RAW_ROOT = PROJECT_ROOT / "data" / "raw" / "archive (2)" / "data3a"
OUT_ROOT = PROJECT_ROOT / "data" / "processed" / "archive2"
TARGET_SIZE = (224, 224)

CLASSES = ["01-minor", "02-moderate", "03-severe"]

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_ROOT exists:", RAW_ROOT.exists())
print("RAW_ROOT:", RAW_ROOT)
print("OUT_ROOT:", OUT_ROOT)
print("TARGET_SIZE:", TARGET_SIZE)

if not RAW_ROOT.exists():
    raise FileNotFoundError("RAW_ROOT not found. Check archive (2)/data3a path.")

# ================= PREPROCESS =================
def preprocess_split(split_name):
    src_base = RAW_ROOT / split_name
    dst_base = OUT_ROOT / ("train" if split_name == "training" else "val")

    processed = skipped = 0
    exts = {".jpg", ".jpeg", ".png"}

    print(f"\n--- {split_name.upper()} ---")

    for cls in CLASSES:
        src_dir = src_base / cls
        dst_dir = dst_base / cls
        dst_dir.mkdir(parents=True, exist_ok=True)

        imgs = [p for p in src_dir.iterdir() if p.suffix.lower() in exts]
        print(f"{cls}: found {len(imgs)} images")

        for img_path in imgs:
            try:
                img = Image.open(img_path).convert("RGB")
                img = img.resize(TARGET_SIZE)
                img.save(dst_dir / img_path.name, format="JPEG", quality=95)
                processed += 1
            except Exception as e:
                skipped += 1

    return processed, skipped

# ================= RUN =================
print("\nPreprocessing TRAIN...")
train_p, train_s = preprocess_split("training")

print("\nPreprocessing VAL...")
val_p, val_s = preprocess_split("validation")

print("\n.. DONE")
print(f"Train processed: {train_p} | skipped: {train_s}")
print(f"Val processed  : {val_p} | skipped: {val_s}")
print("Saved to:", OUT_ROOT)


PROJECT_ROOT: C:\Users\User\Desktop\Vehicle_Damage_Detection
RAW_ROOT exists: True
RAW_ROOT: C:\Users\User\Desktop\Vehicle_Damage_Detection\data\raw\archive (2)\data3a
OUT_ROOT: C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\archive2
TARGET_SIZE: (224, 224)

Preprocessing TRAIN...

--- TRAINING ---
01-minor: found 452 images
02-moderate: found 463 images
03-severe: found 468 images

Preprocessing VAL...

--- VALIDATION ---
01-minor: found 82 images
02-moderate: found 75 images
03-severe: found 91 images

.. DONE
Train processed: 1383 | skipped: 0
Val processed  : 248 | skipped: 0
Saved to: C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\archive2


In [2]:
from pathlib import Path
import os

print("Current working directory:", os.getcwd())

p = Path("../../data/processed/archive2")

print("\nChecking path:", p)
print("Exists:", p.exists())

if p.exists():
    print("\nTop-level contents:")
    for x in p.iterdir():
        print("-", x.name)


Current working directory: c:\Users\User\Desktop\Vehicle_Damage_Detection\notebooks\02_preprocess

Checking path: ..\..\data\processed\archive2
Exists: True

Top-level contents:
- train
- val


In [13]:
from pathlib import Path

ROOT = Path("../../data/processed/archive2")  # correct path from notebooks/02_preprocess

train_root = ROOT / "train"
val_root   = ROOT / "val"

classes = sorted([p.name for p in train_root.iterdir() if p.is_dir()])

def count_images(folder: Path):
    exts = ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]
    total = 0
    for e in exts:
        total += len(list(folder.glob(e)))
    return total

print(" VERIFY:", ROOT.resolve())
print("-" * 50)
print("Classes found:", classes)

print("\nTRAIN counts:")
train_total = 0
for c in classes:
    n = count_images(train_root / c)
    train_total += n
    print(f"{c:<12}: {n}")

print("\nVAL counts:")
val_total = 0
for c in classes:
    n = count_images(val_root / c)
    val_total += n
    print(f"{c:<12}: {n}")

print("\nTOTALS:")
print("Train total:", train_total)
print("Val total  :", val_total)
print("Overall    :", train_total + val_total)


 VERIFY: C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\archive2
--------------------------------------------------
Classes found: ['01-minor', '02-moderate', '03-severe']

TRAIN counts:
01-minor    : 904
02-moderate : 926
03-severe   : 936

VAL counts:
01-minor    : 164
02-moderate : 150
03-severe   : 182

TOTALS:
Train total: 2766
Val total  : 496
Overall    : 3262
